# Exoplanet Filtering for Earth-like Candidates
---
The goal of this notebook is to filter exoplanets based on criteria that approximate Earth-like conditions

### Definition of Earth-like Candidates

In this analysis, Earth-like planets are defined using the following numerical criteria:

- Planet radius: 0.5 - 1.5 Earth radii
- Planet mass: 0.5 - 5 Earth masses
- Equilibrium temperature: within habitable range (200K – 320K)
- Host star age: approximately 4 - 5 Gyr (similar to the Sun)

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import scipy.stats as st

In [3]:
# Load the dataset
data = pd.read_csv('../data/nasa_exoplanet_intelligence.csv')
data

,planet_name,host_star,n_stars,n_planets,discovery_method,disc_year,disc_facility,orbital_period_days,planet_radius_earth,planet_mass_earth,...,ra,dec,controversial_flag,planet_type,habitable_zone_flag,multi_planet_system,is_recent_discovery,dist_category,star_type,orbital_period_cat
0,Kepler-1167 b,Kepler-1167,1,1,Transit,2016.0,Kepler,1.003934,1.710000,3.570000,...,298.302660,47.693965,0,Super-Earth,False,False,False,Far(500-2kpc),K-type,Short(1-10d)
1,Kepler-1740 b,Kepler-1740,1,1,Transit,2021.0,Kepler,8.172400,3.323214,11.000000,...,293.873663,38.922455,0,Mini-Neptune,False,False,True,Far(500-2kpc),G-type(Sun-like),Short(1-10d)
2,Kepler-1581 b,Kepler-1581,1,1,Transit,2016.0,Kepler,6.283855,0.800000,0.437000,...,287.371320,39.603623,0,Sub-Earth,False,False,False,Mid(100-500pc),F-type,Short(1-10d)
3,Kepler-644 b,Kepler-644,1,1,Transit,2016.0,Kepler,3.173917,3.150000,10.100000,...,295.475702,43.493112,0,Mini-Neptune,False,False,False,Far(500-2kpc),F-type,Short(1-10d)
4,Kepler-1752 b,Kepler-1752,1,1,Transit,2021.0,Kepler,56.358501,4.540605,18.700000,...,290.854140,51.222743,0,Neptune-like,False,False,True,Far(500-2kpc),G-type(Sun-like),Medium(10-100d)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6145,KMT-2024-BLG-1870L b,KMT-2024-BLG-1870L,1,1,Microlensing,2026.0,KMTNet,NaN,13.800000,336.898111,...,272.465333,-26.882889,0,Gas Giant,False,False,True,Distant(2k+pc),Unknown,Unknown
6146,TOI-2267 b,TOI-2267 A,2,3,Transit,2025.0,Transiting Exoplanet Survey Satellite (TESS),2.289090,1.000000,0.972000,...,65.061496,84.900824,0,Super-Earth,False,True,True,Nearby(<100pc),M-type(Red Dwarf),Short(1-10d)
6147,TOI-813 b,TOI-813,1,1,Transit,2020.0,Transiting Exoplanet Survey Satellite (TESS),83.891100,6.710000,36.400000,...,72.694010,-60.905461,0,Gas Giant,False,False,True,Mid(100-500pc),G-type(Sun-like),Medium(10-100d)
6148,LHS 1903 b,LHS 1903,1,4,Transit,2026.0,Transiting Exoplanet Survey Satellite (TESS),2.155510,1.382000,3.280000,...,107.865826,48.327933,0,Super-Earth,False,True,True,Nearby(<100pc),M-type(Red Dwarf),Short(1-10d)


In [4]:
in_habitat_zone = data[data['habitable_zone_flag'] == True]
in_habitat_zone

,planet_name,host_star,n_stars,n_planets,discovery_method,disc_year,disc_facility,orbital_period_days,planet_radius_earth,planet_mass_earth,...,ra,dec,controversial_flag,planet_type,habitable_zone_flag,multi_planet_system,is_recent_discovery,dist_category,star_type,orbital_period_cat
7,Kepler-263 c,Kepler-263,1,2,Transit,2014.0,Kepler,47.332773,2.470,6.66,...,292.469802,37.567794,0,Mini-Neptune,True,True,False,Far(500-2kpc),G-type(Sun-like),Medium(10-100d)
8,Kepler-1101 b,Kepler-1101,1,1,Transit,2016.0,Kepler,81.315106,2.470,6.66,...,294.569354,45.671162,0,Mini-Neptune,True,False,False,Far(500-2kpc),G-type(Sun-like),Medium(10-100d)
11,Kepler-560 b,Kepler-560,2,1,Transit,2016.0,Kepler,18.477644,1.720,3.61,...,300.206644,45.018240,0,Super-Earth,True,False,False,Mid(100-500pc),M-type(Red Dwarf),Medium(10-100d)
23,Kepler-1549 b,Kepler-1549,1,1,Transit,2016.0,Kepler,214.886545,2.570,7.13,...,286.052440,39.096028,0,Mini-Neptune,True,False,False,Far(500-2kpc),G-type(Sun-like),Long(100-365d)
33,Kepler-152 c,Kepler-152,1,2,Transit,2014.0,Kepler,88.255055,2.390,6.30,...,286.865383,41.988957,0,Mini-Neptune,True,True,False,Mid(100-500pc),K-type,Medium(10-100d)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6112,KOI-351 g,KOI-351,1,8,Transit,2013.0,Kepler,210.735140,7.718,15.00,...,284.433464,49.305124,0,Gas Giant,True,True,False,Far(500-2kpc),F-type,Long(100-365d)
6120,WASP-47 c,WASP-47,1,4,Radial Velocity,2015.0,La Silla Observatory,589.570000,NaN,447.00,...,331.203092,-12.019067,0,Unknown,True,True,False,Mid(100-500pc),G-type(Sun-like),Very-Long(365d+)
6125,GJ 411 b,GJ 411,1,2,Radial Velocity,2019.0,Haute-Provence Observatory,12.939400,1.450,2.69,...,165.834471,35.972317,0,Super-Earth,True,True,False,Nearby(<100pc),K-type,Medium(10-100d)
6129,HD 192310 b,HD 192310,1,2,Radial Velocity,2011.0,La Silla Observatory,74.720000,4.270,16.90,...,303.828471,-27.033756,0,Neptune-like,True,True,False,Nearby(<100pc),K-type,Medium(10-100d)


In [5]:
# Star Age Filtering

# In this step, we filter planets that orbit stars with an age similar to that of the Sun (4–5 Gyr).

# This criterion aims to select systems that have had sufficient time for potential planetary and biological evolution, making them more comparable to Earth-like conditions.

MIN_STAR_AGE_GYR = 4
MAX_STAR_AGE_GYR = 5

star_age_similar_to_the_sun_age = in_habitat_zone[(in_habitat_zone['star_age_gyr'] > MIN_STAR_AGE_GYR)
                                                  & (in_habitat_zone['star_age_gyr'] < MAX_STAR_AGE_GYR)]
star_age_similar_to_the_sun_age

,planet_name,host_star,n_stars,n_planets,discovery_method,disc_year,disc_facility,orbital_period_days,planet_radius_earth,planet_mass_earth,...,ra,dec,controversial_flag,planet_type,habitable_zone_flag,multi_planet_system,is_recent_discovery,dist_category,star_type,orbital_period_cat
8,Kepler-1101 b,Kepler-1101,1,1,Transit,2016.0,Kepler,81.315106,2.47,6.66,...,294.569354,45.671162,0,Mini-Neptune,True,False,False,Far(500-2kpc),G-type(Sun-like),Medium(10-100d)
11,Kepler-560 b,Kepler-560,2,1,Transit,2016.0,Kepler,18.477644,1.72,3.61,...,300.206644,45.018240,0,Super-Earth,True,False,False,Mid(100-500pc),M-type(Red Dwarf),Medium(10-100d)
23,Kepler-1549 b,Kepler-1549,1,1,Transit,2016.0,Kepler,214.886545,2.57,7.13,...,286.052440,39.096028,0,Mini-Neptune,True,False,False,Far(500-2kpc),G-type(Sun-like),Long(100-365d)
319,TOI-237 b,TOI-237,1,1,Transit,2020.0,Transiting Exoplanet Survey Satellite (TESS),5.436098,1.44,2.67,...,353.243538,-29.416487,0,Super-Earth,True,False,True,Nearby(<100pc),M-type(Red Dwarf),Short(1-10d)
437,HD 176986 d,HD 176986,1,3,Radial Velocity,2026.0,Multiple Observatories,61.376000,2.49,6.76,...,285.773914,-11.044941,0,Mini-Neptune,True,True,True,Nearby(<100pc),K-type,Medium(10-100d)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5802,Kepler-438 b,Kepler-438,1,1,Transit,2015.0,Kepler,35.233190,1.12,1.46,...,281.645724,41.951069,0,Super-Earth,True,False,False,Mid(100-500pc),K-type,Medium(10-100d)
5849,Kepler-1489 b,Kepler-1489,1,2,Transit,2016.0,Kepler,82.294751,1.77,3.78,...,281.295798,44.465406,0,Mini-Neptune,True,True,False,Far(500-2kpc),G-type(Sun-like),Medium(10-100d)
5859,Kepler-1410 b,Kepler-1410,1,1,Transit,2016.0,Kepler,60.866168,1.78,3.82,...,290.510293,38.743501,0,Mini-Neptune,True,False,False,Mid(100-500pc),K-type,Medium(10-100d)
6009,Kepler-540 b,Kepler-540,1,1,Transit,2016.0,Kepler,172.704978,2.71,7.80,...,290.624995,44.873776,0,Mini-Neptune,True,False,False,Mid(100-500pc),G-type(Sun-like),Long(100-365d)


In [6]:
# Host Star Radius Filtering

# In this step, we filter planets whose host stars have a radius close to that of the Sun.

# The selected range (−0.5 to +0.5 in relative units) represents stars with sizes comparable to the Sun, aiming to identify systems with similar stellar conditions to our Solar System.

MIN_STAR_RADIUS_SUN = -0.5
MAX_STAR_RADIUS_SUN = 0.5

star_radius_to_sun = star_age_similar_to_the_sun_age[
    (star_age_similar_to_the_sun_age['star_radius_sun'] >= MIN_STAR_RADIUS_SUN) &
    (star_age_similar_to_the_sun_age['star_radius_sun'] <= MAX_STAR_RADIUS_SUN)
]
star_radius_to_sun

,planet_name,host_star,n_stars,n_planets,discovery_method,disc_year,disc_facility,orbital_period_days,planet_radius_earth,planet_mass_earth,...,ra,dec,controversial_flag,planet_type,habitable_zone_flag,multi_planet_system,is_recent_discovery,dist_category,star_type,orbital_period_cat
11,Kepler-560 b,Kepler-560,2,1,Transit,2016.0,Kepler,18.477644,1.72,3.61,...,300.206644,45.018240,0,Super-Earth,True,False,False,Mid(100-500pc),M-type(Red Dwarf),Medium(10-100d)
319,TOI-237 b,TOI-237,1,1,Transit,2020.0,Transiting Exoplanet Survey Satellite (TESS),5.436098,1.44,2.67,...,353.243538,-29.416487,0,Super-Earth,True,False,True,Nearby(<100pc),M-type(Red Dwarf),Short(1-10d)
450,Kepler-369 c,Kepler-369,1,2,Transit,2014.0,Kepler,14.871572,1.41,2.57,...,293.674332,47.908473,0,Super-Earth,True,True,False,Mid(100-500pc),M-type(Red Dwarf),Medium(10-100d)
820,TOI-1266 d,TOI-1266,1,3,Transit Timing Variations,2025.0,Transiting Exoplanet Survey Satellite (TESS),32.509000,1.74,3.68,...,197.996583,65.833697,0,Super-Earth,True,True,True,Nearby(<100pc),M-type(Red Dwarf),Medium(10-100d)
891,Kepler-296 e,Kepler-296,2,5,Transit,2014.0,Kepler,34.142110,1.53,2.96,...,286.540028,49.437261,0,Super-Earth,True,True,False,Mid(100-500pc),K-type,Medium(10-100d)
892,Kepler-296 f,Kepler-296,2,5,Transit,2014.0,Kepler,63.336270,1.80,3.89,...,286.540028,49.437261,0,Mini-Neptune,True,True,False,Mid(100-500pc),K-type,Medium(10-100d)
2447,TOI-2136 b,TOI-2136,1,1,Transit,2022.0,Transiting Exoplanet Survey Satellite (TESS),7.851928,2.19,6.37,...,281.176343,36.563132,0,Mini-Neptune,True,False,True,Nearby(<100pc),M-type(Red Dwarf),Short(1-10d)
2813,Kepler-54 c,Kepler-54,1,3,Transit,2012.0,Kepler,12.071725,1.23,0.70,...,294.773925,43.056198,0,Super-Earth,True,True,False,Mid(100-500pc),K-type,Medium(10-100d)
3696,Kepler-676 b,Kepler-676,1,1,Transit,2016.0,Kepler,11.598222,3.04,9.48,...,296.222026,50.287363,0,Mini-Neptune,True,False,False,Mid(100-500pc),K-type,Medium(10-100d)
3778,Kepler-1646 b,Kepler-1646,1,1,Transit,2016.0,Kepler,4.485584,1.23,2.04,...,287.526486,42.912936,0,Super-Earth,True,False,False,Mid(100-500pc),M-type(Red Dwarf),Short(1-10d)


In [7]:
# Host Star Mass Filtering

# In this step, we further filter planets whose host stars have a mass close to that of the Sun.

# The selected range (−0.5 to +0.5 in relative units) targets Sun-like stars, ensuring that both stellar size and mass are comparable to our Solar System conditions.

MIN_STAR_MASS_SUN = -0.5
MAX_STAR_MASS_SUN = 0.5

star_mass_to_sun = star_radius_to_sun[(star_radius_to_sun['star_mass_sun'] > MIN_STAR_MASS_SUN)
                                      & (star_radius_to_sun['star_mass_sun'] < MAX_STAR_MASS_SUN)]
star_mass_to_sun

,planet_name,host_star,n_stars,n_planets,discovery_method,disc_year,disc_facility,orbital_period_days,planet_radius_earth,planet_mass_earth,...,ra,dec,controversial_flag,planet_type,habitable_zone_flag,multi_planet_system,is_recent_discovery,dist_category,star_type,orbital_period_cat
11,Kepler-560 b,Kepler-560,2,1,Transit,2016.0,Kepler,18.477644,1.72,3.61,...,300.206644,45.018240,0,Super-Earth,True,False,False,Mid(100-500pc),M-type(Red Dwarf),Medium(10-100d)
319,TOI-237 b,TOI-237,1,1,Transit,2020.0,Transiting Exoplanet Survey Satellite (TESS),5.436098,1.44,2.67,...,353.243538,-29.416487,0,Super-Earth,True,False,True,Nearby(<100pc),M-type(Red Dwarf),Short(1-10d)
820,TOI-1266 d,TOI-1266,1,3,Transit Timing Variations,2025.0,Transiting Exoplanet Survey Satellite (TESS),32.509000,1.74,3.68,...,197.996583,65.833697,0,Super-Earth,True,True,True,Nearby(<100pc),M-type(Red Dwarf),Medium(10-100d)
891,Kepler-296 e,Kepler-296,2,5,Transit,2014.0,Kepler,34.142110,1.53,2.96,...,286.540028,49.437261,0,Super-Earth,True,True,False,Mid(100-500pc),K-type,Medium(10-100d)
892,Kepler-296 f,Kepler-296,2,5,Transit,2014.0,Kepler,63.336270,1.80,3.89,...,286.540028,49.437261,0,Mini-Neptune,True,True,False,Mid(100-500pc),K-type,Medium(10-100d)
2447,TOI-2136 b,TOI-2136,1,1,Transit,2022.0,Transiting Exoplanet Survey Satellite (TESS),7.851928,2.19,6.37,...,281.176343,36.563132,0,Mini-Neptune,True,False,True,Nearby(<100pc),M-type(Red Dwarf),Short(1-10d)
3778,Kepler-1646 b,Kepler-1646,1,1,Transit,2016.0,Kepler,4.485584,1.23,2.04,...,287.526486,42.912936,0,Super-Earth,True,False,False,Mid(100-500pc),M-type(Red Dwarf),Short(1-10d)
3913,L 98-59 e,L 98-59,1,5,Radial Velocity,2021.0,Multiple Observatories,12.827800,1.49,2.82,...,124.532860,-68.314466,0,Super-Earth,True,True,True,Nearby(<100pc),M-type(Red Dwarf),Medium(10-100d)
3914,L 98-59 f,L 98-59,1,5,Radial Velocity,2025.0,Multiple Facilities,23.064000,1.48,2.80,...,124.532860,-68.314466,0,Super-Earth,True,True,True,Nearby(<100pc),M-type(Red Dwarf),Medium(10-100d)
4898,HN Lib b,HN Lib,1,1,Radial Velocity,2023.0,Calar Alto Observatory,36.116000,2.20,5.46,...,218.568482,-12.517006,0,Mini-Neptune,True,False,True,Nearby(<100pc),M-type(Red Dwarf),Medium(10-100d)


In [8]:
# Planet Radius Filtering

# In this step, we filter planets with a radius similar to that of Earth (0.5–1.5 Earth radii).

# This range targets terrestrial-sized planets, which are more likely to have solid surfaces and conditions comparable to Earth.

MIN_PLANET_RADIUS_EARTH = 0.5
MAX_PLANET_RADIUS_EARTH = 1.5

planet_radius_to_earth = star_mass_to_sun[(star_mass_to_sun['planet_radius_earth'] > MIN_PLANET_RADIUS_EARTH)
                                          & (star_mass_to_sun['planet_radius_earth'] < MAX_PLANET_RADIUS_EARTH)]
planet_radius_to_earth

,planet_name,host_star,n_stars,n_planets,discovery_method,disc_year,disc_facility,orbital_period_days,planet_radius_earth,planet_mass_earth,...,ra,dec,controversial_flag,planet_type,habitable_zone_flag,multi_planet_system,is_recent_discovery,dist_category,star_type,orbital_period_cat
319,TOI-237 b,TOI-237,1,1,Transit,2020.0,Transiting Exoplanet Survey Satellite (TESS),5.436098,1.44,2.67,...,353.243538,-29.416487,0,Super-Earth,True,False,True,Nearby(<100pc),M-type(Red Dwarf),Short(1-10d)
3778,Kepler-1646 b,Kepler-1646,1,1,Transit,2016.0,Kepler,4.485584,1.23,2.04,...,287.526486,42.912936,0,Super-Earth,True,False,False,Mid(100-500pc),M-type(Red Dwarf),Short(1-10d)
3913,L 98-59 e,L 98-59,1,5,Radial Velocity,2021.0,Multiple Observatories,12.827800,1.49,2.82,...,124.532860,-68.314466,0,Super-Earth,True,True,True,Nearby(<100pc),M-type(Red Dwarf),Medium(10-100d)
3914,L 98-59 f,L 98-59,1,5,Radial Velocity,2025.0,Multiple Facilities,23.064000,1.48,2.80,...,124.532860,-68.314466,0,Super-Earth,True,True,True,Nearby(<100pc),M-type(Red Dwarf),Medium(10-100d)


In [18]:
# Planet Mass Filtering

# In this step, we filter planets with a mass similar to that of Earth (0.5–5 Earth masses).

# This criterion helps exclude low-density gas giants and focus on planets that are more likely to have a rocky composition, improving the reliability of identifying Earth-like candidates.

MIN_PLANET_MASS = 0.5
MAX_PLANET_MASS = 5

planet_mass_to_earth = planet_radius_to_earth[
    (planet_radius_to_earth['planet_mass_earth'] >= MIN_PLANET_MASS) &
    (planet_radius_to_earth['planet_mass_earth'] <= MAX_PLANET_MASS)
]
planet_mass_to_earth

,planet_name,host_star,n_stars,n_planets,discovery_method,disc_year,disc_facility,orbital_period_days,planet_radius_earth,planet_mass_earth,...,ra,dec,controversial_flag,planet_type,habitable_zone_flag,multi_planet_system,is_recent_discovery,dist_category,star_type,orbital_period_cat
319,TOI-237 b,TOI-237,1,1,Transit,2020.0,Transiting Exoplanet Survey Satellite (TESS),5.436098,1.44,2.67,...,353.243538,-29.416487,0,Super-Earth,True,False,True,Nearby(<100pc),M-type(Red Dwarf),Short(1-10d)
3778,Kepler-1646 b,Kepler-1646,1,1,Transit,2016.0,Kepler,4.485584,1.23,2.04,...,287.526486,42.912936,0,Super-Earth,True,False,False,Mid(100-500pc),M-type(Red Dwarf),Short(1-10d)
3913,L 98-59 e,L 98-59,1,5,Radial Velocity,2021.0,Multiple Observatories,12.827800,1.49,2.82,...,124.532860,-68.314466,0,Super-Earth,True,True,True,Nearby(<100pc),M-type(Red Dwarf),Medium(10-100d)
3914,L 98-59 f,L 98-59,1,5,Radial Velocity,2025.0,Multiple Facilities,23.064000,1.48,2.80,...,124.532860,-68.314466,0,Super-Earth,True,True,True,Nearby(<100pc),M-type(Red Dwarf),Medium(10-100d)


In [19]:
multiplanet_system = planet_mass_to_earth[planet_mass_to_earth['multi_planet_system'] == True]
multiplanet_system

,planet_name,host_star,n_stars,n_planets,discovery_method,disc_year,disc_facility,orbital_period_days,planet_radius_earth,planet_mass_earth,...,ra,dec,controversial_flag,planet_type,habitable_zone_flag,multi_planet_system,is_recent_discovery,dist_category,star_type,orbital_period_cat
3913,L 98-59 e,L 98-59,1,5,Radial Velocity,2021.0,Multiple Observatories,12.8278,1.49,2.82,...,124.53286,-68.314466,0,Super-Earth,True,True,True,Nearby(<100pc),M-type(Red Dwarf),Medium(10-100d)
3914,L 98-59 f,L 98-59,1,5,Radial Velocity,2025.0,Multiple Facilities,23.0640,1.48,2.80,...,124.53286,-68.314466,0,Super-Earth,True,True,True,Nearby(<100pc),M-type(Red Dwarf),Medium(10-100d)


L 98-59 e - https://science.nasa.gov/exoplanet-catalog/l-98-59-e/

L 98-59 f - https://science.nasa.gov/exoplanet-catalog/l-98-59-f/

In the process of analyzing exoplanets, we applied a sequential filtering approach aimed at narrowing down the dataset to the most relevant observations for the research objective.

The first filter we applied was whether the planets are located within the habitable zone. This is the region around a star where conditions may allow the existence of liquid water on the planet’s surface, given a suitable atmosphere. It is important to note that this criterion does not guarantee the presence of life, but only indicates potentially favorable conditions.

Next, we applied additional filters based on the characteristics of the host star. The reasoning behind this is that planets form within a protoplanetary disk surrounding a young star, meaning that the properties of the star significantly influence the formation and evolution of its planetary system. Parameters such as stellar age, temperature, and radius are key factors that can affect the conditions on the planet.

Combining these two types of filters - the planet’s position relative to its star (habitable zone) and the properties of the host star - allows for a more realistic and scientifically grounded selection of objects for analysis.

As expected, with each additional filtering step, the number of planets decreases. This is a natural consequence of applying stricter criteria and indicates a progressive focus on a more specific subset of observations.

However, it is important to acknowledge that each additional filter may lead to a loss of information and can introduce selection bias, which should be considered when interpreting the results.

In the Solar System, gas giants such as Jupiter and Saturn play an important role in the dynamics of small bodies, as they can both reduce and increase the likelihood of impacts on the inner planets. In the L 98-59 system, the absence of similarly massive planets suggests that this gravitational “shielding” mechanism is likely missing, which may result in a different orbital evolution and impact frequency for planets such as L 98-59 e and L 98-59 f.

### Final Result

After applying all filtering criteria, the dataset was reduced to only two exoplanet candidates, both belonging to the same planetary system.

This result highlights how restrictive the defined Earth-like conditions are, significantly narrowing down the number of suitable candidates.

It also suggests that such planets may not be randomly distributed, but rather occur within specific types of planetary systems.

However, this outcome should be interpreted with caution, as the small sample size may also be influenced by data limitations and observational bias.